In [ ]:
import os
import rasterio as rio
import numpy as np
import pandas as pd


from spectral import envi
import matplotlib.pyplot as plt
import matplotlib.path as pth
from pathlib import Path

from scipy.ndimage import binary_erosion
from pyproj import Transformer

import shapefile as shp
import pickle as pk


import Functions
import importlib

importlib.reload(Functions)

In [ ]:
'''
df_height = pd.read_excel(r'/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/App/Data/Ground_Campaign/Flevoland_data/Data_25_fields/Phenology_stages.xlsx',header=0)
display(df_height)
df_height = df_height.map(Functions.handle_ranges)
df_height = df_height.apply(pd.to_numeric, errors='coerce')

df_mediato = df_height.groupby('Code').mean()
df_mediato.to_csv('Averaged_Phenology_Stage.csv',index=True)
display(df_mediato)
'''

In [ ]:
df_avg = pd.read_csv(r'/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/App/Documents/csv/Averaged_crop_height.csv',header=0)
#df_avg = pd.read_csv(r'/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/App/Documents/csv/Averaged_Phenology_Stage.csv',header=0)
df_avg.set_index(df_avg['Code'], inplace=True)
df_avg = df_avg.drop(df_avg.columns[0],axis=1)
df_avg.columns = pd.to_datetime(df_avg.columns)
display(df_avg.index)



In [121]:
df_stats = pd.read_csv('/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/App/Documents/csv/TimeSeries_Coherencia.csv', index_col='Code')
#df_stats = pd.read_csv('/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/App/Documents/csv/TimeSeries_sigma.csv', index_col='Code')

df_stats['Date'] = pd.to_datetime(df_stats['Date'])

display(np.unique(df_stats.index))

array([1545384, 1553694, 1576641, 1601502, 1631664, 1697689, 1697690,
       1697691, 1698168, 1767881, 1824038, 1841223, 1841224, 1841225,
       1889479, 1889482, 1896343, 1936133, 1936134, 1936135, 2041694,
       2081267, 2081268, 2081887, 2088262, 2278835, 2294423])

df_pivot = df_stats.pivot_table(
    index=['Code', 'Date', 'Type', 'Name'], 
    columns='Band', 
    values='mean'
).reset_index()

# 2. Calcoliamo il rapporto VH/VV (in scala lineare o dB)
# Nota: Se i tuoi dati sono in dB, il rapporto VH/VV si ottiene facendo VH - VV
df_pivot['VH_VV_ratio'] = df_pivot['VH'] - df_pivot['VV']
display(df_pivot)

In [ ]:
df_rain = pd.read_csv(r'/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/App/Documents/csv/rainfall_total.csv')
df_rain.set_index(df_rain['date'],inplace=True)
df_rain = df_rain.drop(df_rain.columns[0], axis=1)
df_rain.index = pd.to_datetime(df_rain.index, format='%Y-%m-%d')
display(df_rain)

In [ ]:
prima_corrispondenza = next((d for d in df_rain.index if np.datetime64(d) == df_stats['Date'].iloc[0]), None)

print(f"Data trovata: {prima_corrispondenza}")

In [ ]:
df_filtered = df_rain.loc[prima_corrispondenza:]
plt.plot(df_rain)

In [ ]:
df_weekly = df_filtered.resample('6D', label='left').sum()


In [ ]:
print(df_stats.index)
display(df_avg.index)

grafici con pioggia e crescita

In [122]:
from matplotlib.backends.backend_pdf import PdfPages
# 1. Trasformazione del dizionario in DataFrame (Formato Lungo)
# Definisci il nome del file di uscita

output_folder = Path('/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/App/Documents')
output_folder.mkdir(parents=True, exist_ok=True)

pdf_filename = output_folder / 'TS_Coherence_Height.pdf'

with PdfPages(pdf_filename) as pdf:
    # Assicurati che l'indice di df_avg sia stringa per il matching con 'code'
    df_avg.index = df_avg.index.astype(float).astype(int)
    
    # Raggruppiamo df_stats che contiene i dati radar
    for (code, roi_type, name), group in df_stats.reset_index().groupby(['Code', 'Type', 'Name']):        
        code_str = int(code)
        parcel_ratio = df_stats[df_stats.index == code].sort_values('Date')


        # 1. Controllo: il codice deve essere presente nell'indice delle altezze
        if code_str not in df_avg.index:
            print(f"Skipped {code_str} - {name}")
            continue

        fig, ax1 = plt.subplots(figsize=(12, 6))
        
        # --- ASSE 1: da df_stats ---
        for band in group['Band'].unique():
            band_data = group[group['Band'] == band].sort_values('Date')
            
            # Unica chiamata: Matplotlib gestisce i colori automaticamente (es. VV blu, VH arancione)
            line, = ax1.plot(band_data['Date'], band_data['mean'], 
                            marker='o', markersize=4, linestyle='-', linewidth=2, 
                            label=f'γ {band}', zorder=3)
            
            # L'ombreggiatura seguirà perfettamente il colore della linea
            ax1.fill_between(
                band_data['Date'], 
                band_data['mean'] - band_data['std'], 
                band_data['mean'] + band_data['std'], 
                color=line.get_color(), 
                alpha=0.1, zorder=2
            )

        # --- ASSE 2:
        ax2 = ax1.twinx() 
        
        # Estraiamo la riga corrispondente alla parcella
        grow_series = df_avg.loc[code_str].dropna()
        # Convertiamo i nomi delle colonne (date) in datetime per il filtro temporale
        grow_series.index = pd.to_datetime(grow_series.index)
        
        if not grow_series.empty:
            min_d, max_d = group['Date'].min(), group['Date'].max()
            grow_filt = grow_series.loc[min_d:max_d]
            
            if not grow_filt.empty:
                ax2.plot(grow_filt.index, grow_filt.values, 
                        color='forestgreen', linewidth=2, linestyle='--', label='Height (cm)')
        
        ax2.set_ylabel("Height (cm)", color='forestgreen', fontweight='bold')
        ax2.tick_params(axis='y', labelcolor='forestgreen')
        ax2.set_ylim(bottom=0)

        # --- FINALIZZAZIONE ---
        plt.title(f"Parcel {code_str} ({name}) - {roi_type}")
        ax1.grid(True, alpha=0.3)
        
        # Legenda unificata
        h1, l1 = ax1.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax1.legend(h1 + h2, l1 + l2, loc='upper left', bbox_to_anchor=(1.1, 1))
        
        plt.setp(ax1.get_xticklabels(), rotation=45)
        plt.subplots_adjust(right=0.75, bottom=0.2)
        
        pdf.savefig(fig)
        plt.close(fig)

Skipped 1841224 - DB_A_C


In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
# 1. Trasformazione del dizionario in DataFrame (Formato Lungo)
# Definisci il nome del file di uscita

output_folder = Path('/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/App/Documents')
output_folder.mkdir(parents=True, exist_ok=True)

pdf_filename = output_folder / 'TS_Sigma_Rain_Phenology.pdf'

with PdfPages(pdf_filename) as pdf:
    unique_rois = df_stats.index.unique()

    for (roi, roi_type, crop), group in df_stats.groupby(['Code', 'Name', 'Type']):
        # 1. Prepariamo la figura e gli assi
        fig, ax1 = plt.subplots(figsize=(10, 6))
        
        # Filtriamo i dati ROI
        roi_data = df_stats[df_stats.index == roi]
        
        
        # --- SICUREZZA DATE ---
        # Filtriamo le piogge per mostrare solo lo stesso periodo della coerenza
        min_date = roi_data['Date'].min()
        max_date = roi_data['Date'].max()
        rain_filtered = df_weekly[(df_weekly.index >= min_date) & (df_weekly.index <= max_date)]

        # --- ASSE 2: PIOGGIA (Sotto le linee) ---
        ax2 = ax1.twinx()
        # Usiamo zorder=1 per metterlo sullo sfondo
        ax2.bar(rain_filtered.index, rain_filtered.iloc[:,0], 
                color='skyblue', alpha=0.3, label='Rainfall (mm)', width=0.8, zorder=1)
        
        ax2.set_ylabel("Rainfall (mm)", color='tab:blue', fontsize=12, fontweight='bold')
        ax2.tick_params(axis='y', labelcolor='tab:blue')
        
        # Spazio sopra le barre per non coprire le linee (regola il moltiplicatore se serve)
        if not rain_filtered.empty and rain_filtered.iloc[:,0].max() > 0:
            ax2.set_ylim(0, rain_filtered.iloc[:,0].max() * 2.5)
        ax2.invert_yaxis() 

        # --- ASSE 1: COERENZA (Sopra le barre) ---
        # Portiamo ax1 davanti a ax2
        ax1.set_zorder(ax2.get_zorder() + 1)
        ax1.patch.set_visible(False) # Rende ax1 trasparente per vedere ax2 sotto
        
        for band in roi_data['Band'].unique():
            band_data = roi_data[roi_data['Band'] == band]
            line, = ax1.plot(band_data['Date'], band_data['mean'], 
                             marker='o', markersize=4, linestyle='-', linewidth=2, 
                             label=f'Mean {band}', zorder=3)
            
            ax1.fill_between(
                band_data['Date'], 
                band_data['mean'] - band_data['std'], 
                band_data['mean'] + band_data['std'], 
                color=line.get_color(), 
                alpha=0.1, zorder=2
            )

        ax1.set_ylabel("Sigma", fontsize=12, fontweight='bold')
        ax1.set_xlabel("Data", fontsize=12)

        # --- LEGENDA E LAYOUT ---
        # Uniamo le labels di entrambi gli assi
        h1, l1 = ax1.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        
        # Spostiamo la legenda leggermente più a destra e usiamo subplots_adjust
        ax1.legend(h1 + h2, l1 + l2, loc='upper left', bbox_to_anchor=(1.08, 1), borderaxespad=0.)

        plt.title(f"ROI: {roi} ({roi_type} - {crop})- Coherence vs Rainfall", fontsize=14, pad=20)
        ax1.grid(True, linestyle='--', alpha=0.4)
        
        # Ruotiamo le date sull'asse X
        plt.setp(ax1.get_xticklabels(), rotation=45)
        
        # Invece di tight_layout puro, usiamo margin per far spazio alla legenda
        plt.subplots_adjust(right=0.85, bottom=0.15)
        
        pdf.savefig(fig)
        plt.close(fig)

Grafici raggruppati per CATEGORIA

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# 1. ASSICURIAMOCI CHE LE COLONNE SIANO ACCESSIBILI
# Se 'Code' e 'Type' sono nell'indice, questo li riporta come colonne
df_plot = df_stats.reset_index()

output_folder = Path('/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/App/Documents')
output_folder.mkdir(parents=True, exist_ok=True)
pdf_filename = output_folder / 'Time_Series_Type_Fixed.pdf'

with PdfPages(pdf_filename) as pdf:
    # Usiamo df_plot che ha sicuramente le colonne 'Type', 'Code', 'Band'
    for roi_type, type_group in df_plot.groupby('Type'):
        
        fig, ax1 = plt.subplots(figsize=(12, 7))
        
        # Ora queste righe non daranno più KeyError
        unique_codes = type_group['Code'].unique()
        unique_bands = type_group['Band'].unique()
        
        colors = cm.get_cmap('tab10', len(unique_codes))
        line_styles = {'VV': '-', 'VH': '--', 'COH': '-.'} 

        for i, code in enumerate(unique_codes):
            code_data = type_group[type_group['Code'] == code].sort_values('Date')
            color = colors(i)
            
            for band in unique_bands:
                band_subset = code_data[code_data['Band'] == band]
                
                if not band_subset.empty:
                    style = line_styles.get(str(band).upper(), '-')
                    ax1.plot(band_subset['Date'], band_subset['mean'], 
                             linestyle=style, marker='o', markersize=4,
                             color=color, alpha=0.7,
                             label=f'Parc {code} ({band})')

        # --- FINITURA ---
        ax1.set_title(f"Plant Type: {roi_type}", fontsize=14, fontweight='bold')
        ax1.set_ylabel("Coherence", fontweight='bold')
        ax1.set_ylim(0, 1.1)
        plt.setp(ax1.get_xticklabels(), rotation=45)
        ax1.grid(True, alpha=0.3)
        
        # Legenda posizionata fuori dal grafico
        ax1.legend(loc='upper left', bbox_to_anchor=(1.02, 1), 
                   fontsize='small', ncol=1 if len(unique_codes) < 8 else 2)
        
        plt.subplots_adjust(right=0.75, bottom=0.15)
        
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# 1. Preparazione dati
df_plot = df_stats.reset_index()

output_folder = Path('/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/App/Documents')
output_folder.mkdir(parents=True, exist_ok=True)
pdf_filename = output_folder / 'Time_Series_Type_Aggregated.pdf'

with PdfPages(pdf_filename) as pdf:
    # Raggruppiamo per tipo di raccolto (ogni gruppo diventerà una pagina del PDF)
    for roi_type, type_group in df_plot.groupby('Type'):
        
        fig, ax1 = plt.subplots(figsize=(12, 7))
        
        unique_bands = type_group['Band'].unique()
        # Definiamo colori diversi per le bande (es. VV, VH, COH)
        colors_map = {'VV': 'tab:blue', 'VH': 'tab:orange', 'COH': 'tab:green'}
        line_styles = {'VV': '-', 'VH': '--', 'COH': '-.'}

        ax2 = ax1.twinx()
        # Usiamo zorder=1 per metterlo sullo sfondo
        ax2.bar(df_weekly.index, df_weekly.iloc[:,0], 
                color='skyblue', alpha=0.3, label='Rainfall (mm)', width=0.8, zorder=1)
        
        ax2.set_ylabel("Rainfall (mm)", color='tab:blue', fontsize=12, fontweight='bold')
        ax2.tick_params(axis='y', labelcolor='tab:blue')
        
        # Spazio sopra le barre per non coprire le linee (regola il moltiplicatore se serve)
        if not df_weekly.empty and df_weekly.iloc[:,0].max() > 0:
            ax2.set_ylim(0, df_weekly.iloc[:,0].max() * 2.5)
        ax2.invert_yaxis() 

        for band in unique_bands:
            # Filtriamo per banda e aggreghiamo i dati di TUTTI i 'Code' per quella data
            band_data = type_group[type_group['Band'] == band]
            
            # Calcoliamo media e deviazione standard per ogni data
            # Assumiamo che la colonna con i valori sia 'mean'
            agg_stats = band_data.groupby('Date')['mean'].agg(['mean', 'std']).reset_index()
            agg_stats = agg_stats.sort_values('Date')
            
            if not agg_stats.empty:
                color = colors_map.get(str(band).upper(), None)
                style = line_styles.get(str(band).upper(), '-')
                
                # Plot della linea media
                ax1.plot(agg_stats['Date'], agg_stats['mean'], 
                         linestyle=style, marker='o', markersize=5,
                         color=color, linewidth=2,
                         label=f'Media {band}')
                
                # Plot dell'area della deviazione standard (Ombreggiatura)
                ax1.fill_between(agg_stats['Date'], 
                                 agg_stats['mean'] - agg_stats['std'], 
                                 agg_stats['mean'] + agg_stats['std'], 
                                 color=color, alpha=0.2, 
                                 label=f'Dev. Std {band}')

        # --- FINITURA ---
        ax1.set_title(f"Aggregated Trend: {roi_type}", fontsize=14, fontweight='bold')
        ax1.set_ylabel("Mean Coherence / Value", fontweight='bold')
        ax1.set_xlabel("Date", fontweight='bold')
        
        # Limiti intelligenti: se è coerenza (0-1), altrimenti automatico
        # ax1.set_ylim(0, 1.1) 
        
        plt.setp(ax1.get_xticklabels(), rotation=45)
        ax1.grid(True, linestyle='--', alpha=0.6)
        
        # Legenda
        ax1.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize='medium')
        
        plt.subplots_adjust(right=0.8, bottom=0.15)
        
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)